In [2]:
import os
import sys
import re
import json
from pathlib import Path
from langchain_core.documents import Document
import pypdf
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)
    
# --- 2. Import Your Custom Agents and Utils ---
from crunch_llm.code.agents import * 
from crunch_llm.code.utils import initialize_local_data

/oscar/data/gk/jdtoscan/LLM


/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- 3. Define File Paths and Constants ---
PDF_FILE_PATH = "/users/jdtoscan/data/jdtoscan/LLM/Documents/codes/SciML/Guides/PINNs_Manual_1.pdf" #
DB_PATH = "/users/jdtoscan/data/jdtoscan/LLM/Documents/database_manuals"
COLLECTION_NAME = "PINNs_Manual_v1"
SOURCE_NAME = "PINNs_Manual_1.pdf" # Metadata name

# --- 4. Define the Analyzer Agent Prompt for the Manual ---
analyzer_system_prompt = """
You are an expert on scientific machine learning and scientific computing. Read the provided technical manual
and write a one-sentence summary of its main topic, approximately 50 words long.
Do not add any other explanatory text.
"""


print("--- Initializing Agents and Vector Store ---")

# --- 5. Initialize Google Vector Store ---
# This uses your existing initialize_local_data function
vector_store = initialize_local_data(
    db_path=DB_PATH,
    collection_name=COLLECTION_NAME,
    provider='google', # Use Google
    model_name='models/text-embedding-004' # Google's embedding model
)

# --- 6. Initialize Analyzer Agent ---
analyzer_agent = BasicAgent(
                     model='gemini-2.5-pro', 
                     model_provider='google',
                     system_prompt=analyzer_system_prompt,
                     verbose=False)

--- Initializing Agents and Vector Store ---
Initializing with Google (Gemini). Model: models/text-embedding-004
✅ Vector store initialized successfully from path: /users/jdtoscan/data/jdtoscan/LLM/Documents/database_manuals
   Provider: google, Model: models/text-embedding-004, Collection: PINNs_Manual_v1


In [14]:
all_results = {}
try:
    # --- PDF EXTRACTION STEP ---
    print(f"Reading PDF: {PDF_FILE_PATH}...")
    reader = pypdf.PdfReader(PDF_FILE_PATH)
    full_text_content = ""
    for page in reader.pages:
        full_text_content += page.extract_text() + "\n\n"
    # --- END PDF STEP ---

    line_count = full_text_content.count('\n') + 1
    print(f"Processing: {SOURCE_NAME} ({line_count} lines)...")

    # --- 1. Run Analyzer Agent ONCE on the full file ---
    print("  - Running Analyzer Agent...")
    agent_result = analyzer_agent.run(full_text_content)
    summary_string = agent_result['output']
    
    analysis_result = {
        "summary": summary_string
    }

    # --- 2. THIS IS THE CORRECTED SPLITTING LOGIC ---
    split_pattern = r'(Cell \d+:.*)' # Pattern to split on (and keep)
    parts = re.split(split_pattern, full_text_content)
    
    # re.split gives ['prologue', 'Cell 1: Title', 'Cell 1 Content', 'Cell 2: Title', 'Cell 2 Content', ...]
    
    cleaned_cells = []
    
    # Handle the prologue (anything before "Cell 1")
    if parts[0].strip():
        cleaned_cells.append(parts[0].strip())
        
    # Now, iterate over the rest, taking 2 at a time
    # Start at index 1, step by 2
    for i in range(1, len(parts), 2):
        if i + 1 < len(parts):
            # Combine the title (parts[i]) and its content (parts[i+1])
            full_cell = parts[i] + parts[i+1]
            cleaned_cells.append(full_cell.strip())
        else:
            # Handle a trailing part if it exists
            cleaned_cells.append(parts[i].strip())

    num_cells = len(cleaned_cells)    
    print(f"  - Found {num_cells} cells. Ingesting each as a 'part'...") # This should now be 21

    # --- 3. Prepare common metadata & header info ---
    base_metadata = {
        'source_file': SOURCE_NAME,
        'filename': SOURCE_NAME,
        'type': 'technical-manual',
        **analysis_result 
    }

    comment = "#" 
    analysis_str = json.dumps(analysis_result, indent=2).replace('{', '').replace('}', '').replace('"', '')
    docs_to_add = []
    
    # --- 4. Loop through each cell and create a Document ---
    # (This loop is correct and does not need to change)
    for i, cell_content in enumerate(cleaned_cells):
        part_num = i + 1
        
        cell_description = "File Prologue"
        if cell_content.startswith('Cell '):
            first_line = cell_content.split('\n', 1)[0]
            cell_description = first_line.strip()

        chunk_metadata = base_metadata.copy()
        chunk_metadata['part'] = f"{part_num}/{num_cells}"
        chunk_metadata['cell_description'] = cell_description
        
        header = f"""{comment} DOCUMENT ANALYSIS (PART {part_num} of {num_cells})
{comment} SECTION: {cell_description}
{comment} 
{comment} {analysis_str.replace(chr(10), f'{chr(10)}{comment} ')}
{comment} =======================================================\n\n"""
        
        enhanced_content = header + cell_content
        doc = Document(page_content=enhanced_content, metadata=chunk_metadata)
        docs_to_add.append(doc)

    # --- 5. Add all documents for this file at once ---
    if docs_to_add:
        print(f"  - Adding {len(docs_to_add)} documents to vector store...")
        vector_store.add_documents(docs_to_add)
        
    print(f"✅ Ingested: {SOURCE_NAME} ({num_cells} chunks)")
    all_results[SOURCE_NAME] = {"analysis": analysis_result, "chunks_created": num_cells}

except Exception as e:
    print(f"--- UNHANDLED ERROR analyzing {SOURCE_NAME}: {e} ---")
    all_results[SOURCE_NAME] = {"error": str(e)}

# --- 8. Save and Display Results ---
output_file = 'manual_analysis.json'
print(f"\nAnalysis complete! ✅ Saving all results to {output_file}...")
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=4)
print("Done.")

Reading PDF: /users/jdtoscan/data/jdtoscan/LLM/Documents/codes/SciML/Guides/PINNs_Manual_1.pdf...
Processing: PINNs_Manual_1.pdf (600 lines)...
  - Running Analyzer Agent...
  - Found 20 cells. Ingesting each as a 'part'...
  - Adding 20 documents to vector store...
✅ Ingested: PINNs_Manual_1.pdf (20 chunks)

Analysis complete! ✅ Saving all results to manual_analysis.json...
Done.


In [16]:
# Assuming 'vector_store' is your initialized Chroma object
print("Fetching all documents from the vector store...")

# The .get() method returns a dictionary with IDs, documents, metadatas, etc.
# We specify what information we want to retrieve.
all_entries = vector_store.get(
    include=["metadatas", "documents"] # You can also include "embeddings"
)

# Let's print the first 5 documents found
num_entries = len(all_entries['ids'])
print(f"Found {num_entries} total entries.\n")

for i in range(min(5, num_entries)):
    doc_id = all_entries['ids'][i]
    metadata = all_entries['metadatas'][i]
    content = all_entries['documents'][i]
    
    print(f"--- Entry {i+1} ---")
    print(f"ID: {doc_id}")
    print(f"Source File: {metadata.get('source_file', 'N/A')}")
    print(f"Content Preview: {content[:150]}...") # Print first 150 chars
    print("-" * 20)

Fetching all documents from the vector store...
Found 20 total entries.

--- Entry 1 ---
ID: 12e6481c-6882-40ac-bafc-52c21839ad01
Source File: PINNs_Manual_1.pdf
Content Preview: # DOCUMENT ANALYSIS (PART 1 of 20)
# SECTION: Cell 1: Core Imports and Project Setup
# 
# 
#   summary: This technical manual details a scientific mac...
--------------------
--- Entry 2 ---
ID: 2feefa2f-aedd-4764-b4b4-7a3e90d80a8c
Source File: PINNs_Manual_1.pdf
Content Preview: # DOCUMENT ANALYSIS (PART 2 of 20)
# SECTION: Cell 2: Hyperparameter Configuration (argparse)
# 
# 
#   summary: This technical manual details a scien...
--------------------
--- Entry 3 ---
ID: cfe65ca5-2a5a-40dd-8f7f-534ca2c104c1
Source File: PINNs_Manual_1.pdf
Content Preview: # DOCUMENT ANALYSIS (PART 3 of 20)
# SECTION: Cell 3: Data Loading and Ground Truth Definition
# 
# 
#   summary: This technical manual details a scie...
--------------------
--- Entry 4 ---
ID: 3631717b-4758-46df-a6c0-da1e0693639c
Source File: PINNs_Manual_1

In [15]:
# import os
# import sys
# from pathlib import Path

# # --- Make sure your utils are imported ---
# # (You'll need to adjust this path if necessary)
# file_path = os.getcwd()
# project_root = os.path.dirname(file_path)
# if project_root not in sys.path:
#     sys.path.append(project_root)
    
# from crunch_llm.code.utils import initialize_local_data

# # --- 1. Initialize the correct vector_store (for the MANUAL) ---
# DB_PATH = "/users/jdtoscan/data/jdtoscan/LLM/Documents/database_manuals"
# COLLECTION_NAME = "PINNs_Manual_v1"

# # Initialize the vector store using your unmodified function
# vector_store = initialize_local_data(
#     db_path=DB_PATH,
#     collection_name=COLLECTION_NAME,
#     provider='google', 
#     model_name='models/text-embedding-004'
# )

# if not vector_store:
#     print("Failed to initialize vector store. Exiting.")
#     sys.exit()

# # --- 2. Fetch documents ---
# print("Fetching all documents from the vector store...")
# response = vector_store.get(include=["metadatas"]) # The 'ids' are returned by default
# all_ids = response['ids']
# all_metadatas = response['metadatas']

# # --- 3. Modify the filter logic ---
# target_file = 'PINNs_Manual_1.pdf' # <-- Target the specific file
# ids_to_delete = []

# print(f"Searching for entries with source_file == '{target_file}'...")

# # zip() pairs each id with its corresponding metadata dictionary
# for doc_id, metadata in zip(all_ids, all_metadatas):
#     # Use an exact match for the source file
#     if metadata.get('source_file') == target_file: # <-- Use exact match
#         ids_to_delete.append(doc_id) # Append the correct ID

# # Now 'ids_to_delete' contains the correct list of document IDs
# print(f"Found {len(ids_to_delete)} entries from '{target_file}' to delete.")

# # --- 4. Proceed with the deletion ---
# if ids_to_delete:
#     print(f"Deleting {len(ids_to_delete)} entries...")
#     vector_store.delete(ids=ids_to_delete)
    
#     print("✅ Deletion complete.")
# else:
#     print("No entries found to delete.")